In [3]:
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents.base import Document
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers.weaviate_hybrid_search import WeaviateHybridSearchRetriever
from langchain_community.embeddings.openai import OpenAIEmbeddings
import weaviate
from weaviate.client import WeaviateClient
from weaviate.util import get_valid_uuid
from dotenv import load_dotenv
from uuid import uuid4
import os
from chainlit import logger

In [5]:
load_dotenv()
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
WEAVIATE_API_KEY = os.getenv('WEAVIATE_API_KEY')
WEAVIATE_CLUSTER_ENV = os.getenv('WEAVIATE_CLUSTER_ENV')
print(WEAVIATE_CLUSTER_ENV)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN

https://p7qakbvrqvc49oqx8ixu4g.c0.asia-southeast1.gcp.weaviate.cloud


In [3]:
hf_llm = HuggingFacePipeline.from_model_id(model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0", task="text-generation", 
                                           model_kwargs={'max_length':512})

c:\Users\Admin\.conda\envs\text2sql\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
prompt = PromptTemplate.from_template(template = """\
  <|user|>
  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.
  context:{context}
  question:{question}</s>
  <|assistant|>
  """)

In [8]:
prompt.format(context="CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR",
              question="Show the names of schools with a total budget between 10 to 100")

"  <|user|>\n  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.\n  context:CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR\n  question:Show the names of schools with a total budget between 10 to 100</s>\n  <|assistant|>\n  "

In [12]:
chain = (prompt | hf_llm | StrOutputParser())

In [7]:
print(chain.invoke({'context':"CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR",
                'question':"Show the names of schools with a total budget between 10 to 100"}))

  <|user|>
  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.
  context:CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR
  question:Show the names of schools with a total budget between 10 to 100</s>
  <|assistant|>
  
  SELECT school_name
  FROM budget
  WHERE budgeted BETWEEN 10 AND 100;


In [13]:
res = chain.invoke({'context':"CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR",
                'question':"Show the names of schools with a total budget between 10 to 100"})

In [14]:
res

"  <|user|>\n  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.\n  context:CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR\n  question:Show the names of schools with a total budget between 10 to 100</s>\n  <|assistant|>\n  \n  SELECT school_name\n  FROM budget\n  WHERE budgeted BETWEEN 10 AND 100;"

In [10]:
type(hf_llm), type(chain)

(langchain_huggingface.llms.huggingface_pipeline.HuggingFacePipeline,
 langchain_core.runnables.base.RunnableSequence)

In [15]:
res1 = """<|user|>
  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.
  context:CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR
  question:Show the names of schools with a total budget between 10 to 100</s>
  <|assistant|>
  
  SELECT school_name
  FROM budget
  WHERE budgeted BETWEEN 10 AND 100;"""

In [17]:
res1.split("<|assistant|>")[1].strip()

'SELECT school_name\n  FROM budget\n  WHERE budgeted BETWEEN 10 AND 100;'

VectorDatabase Weaviate

In [78]:
loader = CSVLoader("Data/temp12.csv")
doc = loader.load()
doc[1].metadata

{'source': 'Data/temp12.csv', 'row': 1}

In [51]:
text_splitter = RecursiveCharacterTextSplitter(separators=["/Separate"], keep_separator=False,chunk_size=1000, chunk_overlap=200)
doc1 = text_splitter.split_documents(doc)

In [72]:
doc1[:3]
#len(doc1)

[Document(metadata={'source': 'Data/temp12.csv', 'row': 0}, page_content='question: What is the scheduled start and end date of the CR CRQ000000017039. Please note that the datatype of all Date/Time columns are BIGINT in database and the Date/Time stored in these columns will be in epoch timestamp. Please convert this epcoh timestamp to Date/Time while providing the final result.\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(scheduled_start_date BIGINT, scheduled_end_date BIGINT, infrastructure_change_id VARCHAR)\nBatch:'),
 Document(metadata={'source': 'Data/temp12.csv', 'row': 1}, page_content='question: What is the scheduled start and end date of the CR CRQ000000017059\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(scheduled_start_date BIGINT, cheduled_end_date BIGINT, infrastructure_change_id VARCHAR)\nBatch:'),
 Document(metadata={'source': 'Data/temp12.csv', 'row': 2}, page_content='question: What is the Actual start and end date of the CR CRQ000000017040. Please note that t

In [3]:
'''client = weaviate.Client(url=WEAVIATE_CLUSTER_ENV, auth_client_secret= weaviate.auth.AuthApiKey(WEAVIATE_API_KEY), 
                         additional_headers={"X-OpenAI-Api-Key": OPENAI_API_KEY},)'''
client = weaviate.connect_to_weaviate_cloud(cluster_url=WEAVIATE_CLUSTER_ENV, auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
                                            headers={"X-OpenAI-Api-Key": OPENAI_API_KEY})

In [119]:
client.is_ready()
type(client)

weaviate.client.WeaviateClient

In [84]:
my_collection = client.collections.get("T2SQL")
my_collection.config.get()

_CollectionConfig(name='T2SQL', description=None, generative_config=None, inverted_index_config=_InvertedIndexConfig(bm25=_BM25Config(b=0.75, k1=1.2), cleanup_interval_seconds=60, index_null_state=False, index_property_length=False, index_timestamps=False, stopwords=_StopwordsConfig(preset=<StopwordsPreset.EN: 'en'>, additions=None, removals=None)), multi_tenancy_config=_MultiTenancyConfig(enabled=False, auto_tenant_creation=False, auto_tenant_activation=False), properties=[_Property(name='context', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=_PropertyVectorizerConfig(skip=False, vectorize_property_name=True), vectorizer='text2vec-openai'), _Property(name='question', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=_P

In [115]:
retriever = WeaviateHybridSearchRetriever(client=client, 
                                          index_name="T2SQL", 
                                          alpha=0.5, #param to balance b/w keyword and semantic search
                                          text_key="context", 
                                          attributes=[], 
                                          create_schema_if_missing=True)
type(retriever)

langchain_community.retrievers.weaviate_hybrid_search.WeaviateHybridSearchRetriever

In [86]:
def add_documents(client: WeaviateClient, index_name, text_key, documents: list, **kwargs):
    collection = client.collections.get(index_name)
    with collection.batch.dynamic() as batch:
        id = []
        for i, doc  in enumerate(documents):
            data_row = {text_key: doc.page_content}
            if "uuids" in kwargs:
                _id = kwargs["uuids"][i]
            else:
                _id = get_valid_uuid(uuid4())
            batch.add_object(properties=data_row, uuid=_id)
            id.append(_id)
    return id


In [87]:
ids = add_documents(client=client, index_name="T2SQL", text_key="context", documents=doc)

In [113]:
collection = client.collections.get("T2SQL")
response = collection.query.hybrid(query="When was the CRQ000000017391 created?", alpha=0.25, limit=5)

In [97]:
for o in response.objects:
    print(o.properties)

{'context': 'question: What is the Actual Start date of the CR CRQ000000017281\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(actual_start_date BIGINT, infrastructure_change_id VARCHAR)', 'source': 'Data/temp12.csv', 'question': None, 'row': 11.0}
{'context': 'question: What is the Actual Start date of the CR CRQ000000017281\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(actual_start_date BIGINT, infrastructure_change_id VARCHAR)', 'source': 'Data/temp12.csv', 'question': None, 'row': 11.0}
{'context': 'question: What is the Actual start and end date of the CR CRQ000000017080\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(actual_start_date BIGINT, actual_end_date BIGINT, infrastructure_change_id VARCHAR)\nBatch: ', 'source': None, 'question': None, 'row': None}
{'context': 'question: What is the Actual start and end date of the CR CRQ000000017080\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(actual_start_date BIGINT, actual_end_date BIGINT, infrastructure_change_id VARCHAR)', 

In [102]:
response.objects[0].properties.keys()

dict_keys(['context', 'source', 'question', 'row'])

In [117]:
response.objects[0].properties['context'].split("\n")[1]

'context: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(submit_date BIGINT, infrastructure_change_id VARCHAR)'

logging.Logger